# Naive Bayes on Real Data — Worksheet
### Build a spam filter from scratch, add the log trick, then match scikit-learn

You've learned **Multinomial Naive Bayes with add-1 (Laplace) smoothing** for text. Here you'll turn that math into a real Python **class** and train it on the **SMS Spam Collection** — about 5,570 *real* text messages labelled `spam` or `ham` (not-spam). Then you'll make it numerically safe with the **log trick** and reproduce it with **scikit-learn**.

Look for `# TODO` markers, fill them in, and run the check cells. A check that prints  passed.

Parts A–C use only the Python standard library. Part D uses scikit-learn.

## The algorithm you're implementing (quick reference)

For a message made of words $w_1, w_2, \dots$ and a class $c$ (here `spam` or `ham`):

**1. Prior** — how common each class is:
$$P(c) = \frac{N_c}{N_{docs}}$$

**2. Likelihood (add-1 smoothed):**
$$P(w \mid c) = \frac{\text{count}(w, c) + 1}{(\text{total words in } c) + |V|}$$
where $|V|$ is the vocabulary size (distinct words in training).

**3. Classify** — pick the highest-scoring class:
$$\hat{c} = \arg\max_c \; P(c)\prod_{w \in msg} P(w \mid c)$$

**Rules:** unknown words (never seen in training) are **dropped**; add-1 smoothing keeps every $P(w\mid c)>0$ so a single missing word can't zero out a class.

> Heads-up: real data is **imbalanced** — about 87% of messages are ham. So unlike a toy 50/50 set, the **prior genuinely matters** here.

## Load the real dataset 📱

The `sms_spam.tsv` file has one message per line as `label⇥text`. If it isn't next to the notebook, the cell downloads it. Run it.

##### Multinomial Naive Bayes is a relatively simple linear probabilistic classifier. It doesn't have complex weights or millions of parameters

In [1]:
import os, re, random, urllib.request

URL  = "https://raw.githubusercontent.com/justmarkham/pycon-2016-tutorial/master/data/sms.tsv"
PATH = "sms_spam.tsv"
if not os.path.exists(PATH):
    print("Downloading SMS Spam Collection ...")
    urllib.request.urlretrieve(URL, PATH)

docs, labels = [], []       # docs = SMS string, labels = spam/ham
for line in open(PATH, encoding="utf-8"):
    line = line.rstrip("\n")
    if not line:
        continue
    label, text = line.split("\t", 1)
    labels.append(label)
    docs.append(text)

print(len(docs), "messages loaded")
print("spam:", labels.count("spam"), " ham:", labels.count("ham"))
print("\nexample spam ->", docs[labels.index("spam")][:90], "...")

5574 messages loaded
spam: 747  ham: 4827

example spam -> Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 87121 to re ...


### Shuffle and split into train / test

We fix the random seed so everyone gets the same split (and the same answers).

In [2]:
random.seed(42)   # We shuffle so the model gets a fair, well-balanced mix of data
order = list(range(len(docs)))
random.shuffle(order)
docs   = [docs[i]   for i in order]
labels = [labels[i] for i in order]

cut = int(0.8 * len(docs))      # shows 80/20 split
train_docs,  test_docs   = docs[:cut],   docs[cut:]
train_labels, test_labels = labels[:cut], labels[cut:]
print("train:", len(train_docs), " | test:", len(test_docs))

train: 4459  | test: 1115


---
## Part A — Build the `MultinomialNaiveBayes` class

Real messages have capitals, punctuation and numbers, so the tokenizer (given) lowercases the text and pulls out runs of letters/digits. Everything else is the same math as the reference above — fill in the TODOs.

In [3]:
import re

class MultinomialNaiveBayes:
    def __init__(self, alpha=1):
        self.alpha = alpha                       # add-1 smoothing => alpha = 1
        # It prevents zero-multiplication errors when a word is missing.

    def _tokenize(self, text):
        return re.findall(r"[a-z0-9]+", text.lower())   # given: real tokenizer
        # Cleans up the incoming sentence. (lower case, rmv punctuation)

    def fit(self, docs, labels):       # This method reads all training messages and builds the reference counts needed for probabilities.
        self.classes = sorted(set(labels))
        self.vocab = set()            # ignore the dupes
        self.doc_count   = {c: 0  for c in self.classes}   # docs/sentence per class (for prior)
        self.word_counts = {c: {} for c in self.classes}   # word -> count, per class
        self.total_words = {c: 0  for c in self.classes}   # total tokens, per class
        n_docs = len(docs)

        for text, c in zip(docs, labels):
            tokens = self._tokenize(text)
            self.doc_count[c] += 1
            for w in tokens:   # w is a temporary loop variable that stands for a single word extracted from a text message.

                # TODO 1: add w to self.vocab
                self.vocab.add(w)

                # TODO 2: add 1 to self.word_counts[c][w]  (use .get(w, 0), it may be new)
                self.word_counts[c][w] = self.word_counts[c].get(w, 0) + 1
                # If a word hasn't been seen in this category yet, looking it up normally crashes the code with a KeyError.
                #.get(w, 0) says: "Give me the current count if it exists, otherwise assume 0",

                # TODO 3: add 1 to self.total_words[c]
                self.total_words[c] += 1   # Count every single word token that appears in class c(spam/not spam)
                pass

        self.vocab_size = len(self.vocab)  # denominator updates

        # TODO 4: prior of each class = docs in that class / total docs
        self.priors = {}
        for c in self.classes:
            self.priors[c] = self.doc_count[c] / n_docs        # What percentage of all messages belong to this class?

        return self

    def _likelihood(self, word, c):
        """add-1 smoothed P(word | c)."""
        count = self.word_counts[c].get(word, 0)   # count of words with 0 occurence

        # TODO 5: return (count + alpha) / (total_words[c] + alpha * vocab_size)
        return (count + self.alpha) / (self.total_words[c] + self.alpha * self.vocab_size)       # The Likelihood Formula
        # Adding alpha = 1 makes sure every word has a non-zero probability.

    def predict_one(self, text):
        tokens = self._tokenize(text)
        best_class, best_score = None, -1.0
        # In standard probability, probabilities are positive numbers between 0 and 1,
        # so initializing a baseline at -1.0 works because any real probability (0.0 to 1.0) is guaranteed to be larger

        for c in self.classes:
            score = self.priors[c]               # start from the prior
            for w in tokens:
                if w in self.vocab:              # drop unknown words

                    # TODO 6: multiply score by the likelihood of w given c
                    score *= self._likelihood(w, c)
                    pass

            # TODO 7: if score beats best_score, remember this class
            if score > best_score:
              best_score = score
              best_class = c
            pass
        return best_class

    def predict(self, docs):
        return [self.predict_one(t) for t in docs]

##### the above cell only defines the MultinomialNaiveBayes class and its methods, so it loads the blueprint into memory

A small helper to measure accuracy (given):

In [4]:
# what percentage of msgs it correctly classified

def accuracy(model, docs, labels):
    preds = model.predict(docs)    # Gives the test messages (docs) to your model and it returns spam/not spam
    return sum(p == y for p, y in zip(preds, labels)) / len(labels)

    # Pairs each model prediction with the actual, true answer from labels
    # Returns True if the prediction matches the real label
    # Adding them ( T=1 and F=0) up counts the total number of correct predictions.

###  Check A1 — did `fit` learn sensible things?

In [5]:
model = MultinomialNaiveBayes(alpha=1).fit(train_docs, train_labels)
print("vocabulary size:", model.vocab_size)
print("priors:", {c: round(p, 3) for c, p in model.priors.items()})

# These lines are automated unit tests using Python's assert statement to verify that the fit() method calculated all training parameters properly

assert model.vocab_size > 5000, "real vocab should be in the thousands"
# The training set contains over 4,000 text messages, which realistically span thousands of distinct words. If this number is lower than 5,000, it indicates an issue in _tokenize

assert abs(sum(model.priors.values()) - 1) < 1e-9, "priors must sum to 1"
# probabilities equals 1.

assert model.priors["ham"] > model.priors["spam"], "ham should be the majority class"

print("fit looks correct")

vocabulary size: 7725
priors: {'ham': 0.868, 'spam': 0.132}
fit looks correct


###  Check A2 — how well does it classify unseen messages?

In [6]:
acc = accuracy(model, test_docs, test_labels)
print(f"test accuracy: {acc:.4f}")
assert acc > 0.95, "Naive Bayes should score well above 95% on this dataset"
print("predict works — accuracy above 95%")
# it scores 98% acc on test data (20%)

test accuracy: 0.9830
predict works — accuracy above 95%


#####  About $87 \%$ of this dataset consists of messages (ham), while only roughly $13\%$ is spam

*italicized text*###  Explore — try your filter on messages you write

In [7]:
# This cell tests your trained model on custom, real-world examples and discovers which specific words carry the strongest spam signals.
for msg in ["congratulations you won a free prize call now to claim",
            "hey are we still on for dinner tonight",
            "URGENT your account has won 1000 cash reply now"]:
    print(f"{model.predict_one(msg):>4}  <-  {msg}")

# which words most strongly signal spam?
ratio = {w: model._likelihood(w, "spam") / model._likelihood(w, "ham") for w in model.vocab}
# It loops through every unique word w in your vocabulary and calculates an odds ratio (a comparison score)
print("\nmost spam-indicative words:", sorted(ratio, key=ratio.get, reverse=True)[:8])
# Finding and Printing the Top 8 Winners which indicate spamminess

spam  <-  congratulations you won a free prize call now to claim
 ham  <-  hey are we still on for dinner tonight
spam  <-  URGENT your account has won 1000 cash reply now

most spam-indicative words: ['claim', 'prize', '150p', 'tone', '18', 'guaranteed', 'cs', '1000']


---
## Part B — The log trick (and why real documents need it)

On short SMS your filter is fine. But multiplying many probabilities makes the number shrink fast — on a **longer document** it underflows to `0.0`, and every class ties. Let's build a long "email" by joining several real spam messages and watch it break:

Probabilities are decimal numbers between $0$ and $1$ .When evaluating a short SMS with only $5$ words, multiplying small numbers produces a tiny fraction that the computer can still manage, However, a longer message or an email might have $50$ to $100$ words. Every time you multiply by another tiny fraction, the value shrinks exponentially. Eventually, the number becomes smaller than what computer memory can represent with standard floating-point precision. this issue is called **arithematic underflow** and computer eventually rounds it to 0.0, this will then affect the decision making of the model

In [8]:
long_email = " ".join([t for t, y in zip(train_docs, train_labels) if y == "spam"][:6])
# Glues the 6(spam) texts together into one long paragraph to simulate a full-length email.

print("length in tokens:", len(model._tokenize(long_email)))
# Counts how many words are in this combined text

score = model.priors["spam"]

# Loops through every word in the long email and repeatedly multiplies the score by the word's likelihood.
for w in model._tokenize(long_email):
    if w in model.vocab:
        score *= model._likelihood(w, "spam")
print("plain product score:", score, "  <- 0.0 means it underflowed to nothing")


length in tokens: 165
plain product score: 0.0   <- 0.0 means it underflowed to nothing


**The fix:** add **logarithms** instead of multiplying. `log` turns a product into a sum, keeps the numbers in a safe range, and doesn't change which class wins:

$$\log\Big(P(c)\prod_w P(w\mid c)\Big) = \log P(c) + \sum_w \log P(w\mid c)$$

Since it's the *same model* scored differently, subclass `MultinomialNaiveBayes` and override only `predict_one` — reusing `fit` and `_likelihood` for free (**inheritance**).

In standard probability, probabilities are positive numbers between $0$ and $1$, so initializing a baseline at -1.0 works because any real probability ($0.0$ to $1.0$) is guaranteed to be larger

In [9]:
import math

# this class fixes underflow problem by using log trick

class LogNaiveBayes(MultinomialNaiveBayes):   # using inherwitence( no need to rewrite)
    def predict_one(self, text):      # overriding : It replaces only the scoring logic inside predict_one.
        tokens = self._tokenize(text)

        # In log-space, probabilities range from 0 down to negative infinity
        # guarantees that any real calculated log score, will be strictly greater than the initial value
        best_class, best_score = None, float("-inf")

        for c in self.classes:
            # TODO A: start score at log(prior of c)
            score = math.log(self.priors[c])

            for w in tokens:
                if w in self.vocab:
                    # TODO B: ADD log(likelihood of w given c) to score
                    score += math.log(self._likelihood(w, c))
                    pass
            # TODO C: keep the highest-scoring class
            if score > best_score:
              best_score = score
              best_class = c
            pass
        return best_class

### Check B — just as accurate, and it survives the long document

In [10]:
# This cell validates that the LogNaiveBayes implementation solves the
# arithmetic underflow problem on long documents without sacrificing overall accuracy.

log_model = LogNaiveBayes(alpha=1).fit(train_docs, train_labels)
log_acc = accuracy(log_model, test_docs, test_labels)
print(f"log-space accuracy: {log_acc:.4f}")
print("long email classified as:", log_model.predict_one(long_email))

assert log_acc > 0.95, "log version should be just as accurate"
assert log_model.predict_one(long_email) == "spam", "log version must handle the long email"
print(" log version works and survives the document that underflowed above")

log-space accuracy: 0.9830
long email classified as: spam
✅ log version works and survives the document that underflowed above


---
## Part C — Short questions

1. Why do we **drop unknown words** instead of giving them probability 0?
2. What does add-1 smoothing prevent? What would `alpha = 0` do?
3. About **87%** of messages are ham, so `P(ham) ≈ 0.87`. For a short, neutral message with no strong spam words, which way does that prior tip the decision, and why?
4. Give one real reason the **log trick** matters for longer documents (emails, articles).

In [11]:
# Your answers (1-4):
# 1. A probability of 0 would multiply the entire message's score to absolute zero

# 2. It prevents unseen words within a class from wiping out that class's scorey
#    alpha = 0 leaves no safety net, assigning a zero probability to any message containing an unseen word.

# 3. It tips the decision toward ham because when word evidence is weak or balanced,
#    the class with the higher baseline prior naturally dominates the calculation.
#    and becaus the p(ham) = 0.87% The prior acts as your default assumption in naive- bayes

# 4. Multiplying dozens of fractional probabilities causes floating-point numbers to shrink until they
#    underflow to 0.0, whereas adding logarithms keeps the numbers safely within computer precision limits.


---
## Part D — Now do it with scikit-learn

Real projects use `CountVectorizer` (text → word-count vectors) + `MultinomialNB` (the model you just built). Fill the TODOs, then compare to your own class.

> We set `token_pattern` and `lowercase` to match your `_tokenize`, so both use the exact same vocabulary — otherwise their word lists (and probabilities) would differ slightly.

Machine learning models cannot read raw strings like "win cash now"; they only understand numbers.

CountVectorizer is a converter that turns text into a table of numbers (a word-count matrix).
It works in two steps:

1. Finds all unique words across all your messages to build a master dictionary (vocabulary).
2.  Counts word frequencies for each message. Every message becomes a row, every word becomes a column, and the cells contain how many times that word appears.

Message: "win cash win"

Vocabulary: ["cash", "hello", "win"]

Vector:    [  1   ,    0   ,   2  ]




In [12]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB

vec = CountVectorizer(lowercase=True, token_pattern=r"[a-z0-9]+")   # match our tokenizer

# TODO 1: fit_transform the TRAINING docs into a count matrix
X_train = vec.fit_transform(train_docs)
# X_train becomes a large matrix of numbers representing all training docs(SMS).

# TODO 2: create MultinomialNB(alpha=1) and fit it on X_train, train_labels
clf = MultinomialNB(alpha=1).fit(X_train, train_labels)

# TODO 3: transform the TEST docs and predict
X_test = vec.transform(test_docs)
sk_preds = clf.predict(X_test)

print("first 8 scikit-learn predictions:", list(sk_preds)[:8])

first 8 scikit-learn predictions: [np.str_('ham'), np.str_('ham'), np.str_('ham'), np.str_('spam'), np.str_('spam'), np.str_('spam'), np.str_('ham'), np.str_('ham')]


### ✅ Check D — your class vs. scikit-learn

In [13]:
sk_acc = sum(str(p) == y for p, y in zip(sk_preds, test_labels)) / len(test_labels)
my_preds = log_model.predict(test_docs)
agree = sum(str(a) == b for a, b in zip(sk_preds, my_preds)) / len(sk_preds)

print(f"scikit-learn accuracy      : {sk_acc:.4f}")
print(f"agreement with your model  : {agree:.4f}")
assert sk_acc > 0.95
assert agree > 0.98, "your log model should essentially match scikit-learn"
print("\nYour from-scratch Naive Bayes matches scikit-learn!")

scikit-learn accuracy      : 0.9830
agreement with your model  : 1.0000

✅ Your from-scratch Naive Bayes matches scikit-learn!


---
## 🎯 What you built
- A real `fit` / `predict` **Naive Bayes spam filter** with add-1 smoothing, ~**98% accurate** on real SMS.
- A **log-space** version (via **inheritance**) that survives long documents where the plain product underflows.
- The **same result** from **scikit-learn**, confirming your math.

You now know exactly what `MultinomialNB` does under the hood — it's the class you just wrote.